# Generative Models pada Fashion-MNIST
Perbandingan AE, Sparse AE, VAE, GAN, DCGAN, dan DDPM.

In [ ]:
# - Workspace setup
from pathlib import Path
import os

if not Path('src/models.py').exists() and Path('Generative_Models/src/models.py').exists():
    os.chdir('Generative_Models')
assert Path('src/models.py').exists(), 'Buka notebook dari folder Generative_Models.'
print('Workspace:', Path.cwd())

In [ ]:
# - Models setup
import math
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
# - Encoder
def encoder():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 64), nn.ReLU())

In [ ]:
# - Decoder
def decoder():
    return nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid(), nn.Unflatten(1, (1, 28, 28)))

In [ ]:
# - Autoencoder
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = encoder()
        self.decoder = decoder()

    def forward(self, images):
        latent = self.encoder(images)
        return self.decoder(latent), latent

In [ ]:
# - Vae
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU())
        self.mean = nn.Linear(256, 64)
        self.logvar = nn.Linear(256, 64)
        self.decoder = decoder()

    def encode(self, images):
        hidden = self.features(images)
        return self.mean(hidden), self.logvar(hidden).clamp(-20, 20)

    def forward(self, images):
        mean, logvar = self.encode(images)
        latent = mean + torch.exp(0.5 * logvar) * torch.randn_like(mean)
        return self.decoder(latent), mean, logvar

In [ ]:
# - Classifier
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = encoder()
        self.head = nn.Linear(64, 10)

    def forward(self, images):
        return self.head(self.encoder(images))

In [ ]:
# - Generator
class Generator(nn.Module):
    def __init__(self, convolutional=False):
        super().__init__()
        self.latent_dim = 100 if convolutional else 64
        if convolutional:
            self.net = nn.Sequential(
                nn.Unflatten(1, (100, 1, 1)),
                nn.ConvTranspose2d(100, 128, 7, bias=False), nn.BatchNorm2d(128), nn.ReLU(),
                nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
                nn.ConvTranspose2d(64, 1, 4, 2, 1), nn.Tanh(),
            )
        else:
            self.net = nn.Sequential(
                nn.Linear(64, 256), nn.LeakyReLU(0.2),
                nn.Linear(256, 512), nn.LeakyReLU(0.2),
                nn.Linear(512, 784), nn.Tanh(), nn.Unflatten(1, (1, 28, 28)),
            )

    def forward(self, latent):
        return self.net(latent)

In [ ]:
# - Discriminator
class Discriminator(nn.Module):
    def __init__(self, convolutional=False):
        super().__init__()
        if convolutional:
            self.net = nn.Sequential(
                nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),
                nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
                nn.Flatten(), nn.Linear(128 * 7 * 7, 1),
            )
        else:
            self.net = nn.Sequential(
                nn.Flatten(), nn.Linear(784, 512), nn.LeakyReLU(0.2),
                nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Linear(256, 1),
            )

    def forward(self, images):
        return self.net(images).flatten()

In [ ]:
# - Timeblock
class TimeBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(4, out_channels)
        self.time = nn.Linear(64, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(4, out_channels)
        self.skip = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, images, time_embedding):
        hidden = F.silu(self.norm1(self.conv1(images)))
        hidden = hidden + self.time(time_embedding)[:, :, None, None]
        return F.silu(self.norm2(self.conv2(hidden)) + self.skip(images))

In [ ]:
# - Noisepredictor
class NoisePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.time = nn.Sequential(nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 64))
        self.down1 = TimeBlock(1, 16)
        self.down2 = TimeBlock(16, 32)
        self.middle = TimeBlock(32, 64)
        self.up2 = TimeBlock(96, 32)
        self.up1 = TimeBlock(48, 16)
        self.output = nn.Conv2d(16, 1, 1)

    def forward(self, images, timesteps):
        frequency = torch.exp(-math.log(10000) * torch.arange(32, device=images.device) / 31)
        angles = timesteps.float()[:, None] * frequency[None]
        time_embedding = self.time(torch.cat([angles.sin(), angles.cos()], dim=1))
        first = self.down1(images, time_embedding)
        second = self.down2(F.avg_pool2d(first, 2), time_embedding)
        middle = self.middle(F.avg_pool2d(second, 2), time_embedding)
        hidden = self.up2(torch.cat([F.interpolate(middle, size=14, mode='nearest'), second], 1), time_embedding)
        hidden = self.up1(torch.cat([F.interpolate(hidden, size=28, mode='nearest'), first], 1), time_embedding)
        return self.output(hidden)

In [ ]:
# - Diffusion
class Diffusion:
    def __init__(self, device, steps=200):
        self.steps = steps
        time = torch.linspace(0, steps, steps + 1, device=device) / steps
        cumulative = torch.cos((time + 0.008) / 1.008 * math.pi / 2).square()
        cumulative = cumulative / cumulative[0]
        self.beta = (1 - cumulative[1:] / cumulative[:-1]).clamp(0.0001, 0.999)
        self.alpha = 1 - self.beta
        self.alpha_bar = self.alpha.cumprod(0)
        previous = F.pad(self.alpha_bar[:-1], (1, 0), value=1)
        self.variance = self.beta * (1 - previous) / (1 - self.alpha_bar)
        self.coef1 = self.beta * previous.sqrt() / (1 - self.alpha_bar)
        self.coef2 = (1 - previous) * self.alpha.sqrt() / (1 - self.alpha_bar)

    def add_noise(self, images, timesteps, noise):
        weight = self.alpha_bar[timesteps, None, None, None]
        return weight.sqrt() * images + (1 - weight).sqrt() * noise

    @torch.no_grad()
    def sample(self, model, count, trajectory=False):
        images = torch.randn(count, 1, 28, 28, device=self.beta.device)
        frames = [images[:1].cpu()]
        for step in reversed(range(self.steps)):
            timesteps = torch.full((count,), step, device=images.device, dtype=torch.long)
            noise = model(images, timesteps)
            original = ((images - (1 - self.alpha_bar[step]).sqrt() * noise) / self.alpha_bar[step].sqrt()).clamp(-1, 1)
            images = self.coef1[step] * original + self.coef2[step] * images
            if step > 0:
                images += self.variance[step].sqrt() * torch.randn_like(images)
            if trajectory and step in [159, 119, 79, 39, 0]:
                frames.append(images[:1].cpu())
        result = (images.clamp(-1, 1) + 1) / 2
        return (result, frames) if trajectory else result

In [ ]:
# - Pipeline setup
import json
import random
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import FashionMNIST
from sklearn.model_selection import train_test_split
CONFIG = {
    'seed': 42,
    'batch_size': 256,  # - Jumlah gambar dalam satu batch
    'latent_dim': 64,
    'sparsity_weight': 0.001,  # - Penalti rata-rata aktivasi laten
    'diffusion_steps': 200,
    'epochs': {'AE': 25, 'SparseAE': 25, 'VAE': 25, 'GAN': 40, 'DCGAN': 40, 'DDPM': 30, 'Classifier': 35, 'FrozenDecoder': 25},
    'fid_samples': 3000,  # - Ukuran sama untuk real dan generated samples
}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path.cwd()
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(exist_ok=True)
torch.set_num_threads(4)

In [ ]:
# - Seed Everything
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [ ]:
# - Synchronize
def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

In [ ]:
# - Prepare Data
def prepare_data():
    config_path = OUTPUT / 'config.json'
    if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
        raise ValueError('Configuration berubah. Arsipkan outputs sebelum memulai eksperimen baru.')
    # - Gabungkan official partitions sebelum split internal 80/10/10
    original_train = FashionMNIST(ROOT / 'data', train=True, download=True)
    original_test = FashionMNIST(ROOT / 'data', train=False, download=True)
    images = torch.cat([original_train.data, original_test.data]).unsqueeze(1).float() / 255
    labels = torch.cat([original_train.targets, original_test.targets])
    indices = np.arange(len(labels))
    train_index, remaining = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels.numpy())
    val_index, test_index = train_test_split(remaining, test_size=0.5, random_state=42, stratify=labels[remaining].numpy())
    np.savez(OUTPUT / 'split_indices.npz', train=train_index, validation=val_index, test=test_index)
    data = {}
    for split, index in [('train', train_index), ('validation', val_index), ('test', test_index)]:
        # - Cache tensor pada GPU agar transfer tidak mendominasi training
        data[split] = TensorDataset(images[index].to(DEVICE), labels[index].to(DEVICE))
    (OUTPUT / 'config.json').write_text(json.dumps(CONFIG, indent=2))
    (OUTPUT / 'hardware.json').write_text(json.dumps({'device': str(DEVICE), 'gpu': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None, 'torch': torch.__version__, 'precision': 'FP32', 'split': {key: len(value) for key, value in data.items()}}, indent=2))
    return data

In [ ]:
# - Loader
def loader(dataset, training=False):
    return DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=training, num_workers=0)

In [ ]:
# - Objective
def objective(name, model, images, labels, classifier=None, diffusion=None):
    if name == 'Classifier':
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        return loss, {'accuracy': (logits.argmax(1) == labels).float().mean()}
    if name == 'FrozenDecoder':
        with torch.no_grad():
            latent = classifier.encoder(images)
        loss = F.mse_loss(model(latent), images)
        return loss, {'reconstruction': loss}
    if name == 'DDPM':
        images = images * 2 - 1
        timestep = torch.randint(diffusion.steps, (len(images),), device=DEVICE)
        noise = torch.randn_like(images)
        prediction = model(diffusion.add_noise(images, timestep, noise), timestep)
        loss = F.mse_loss(prediction, noise)
        return loss, {'noise_mse': loss}
    if name == 'VAE':
        reconstructed, mean, logvar = model(images)
        reconstruction = F.binary_cross_entropy(reconstructed, images, reduction='none').flatten(1).sum(1).mean()
        kl = -0.5 * (1 + logvar - mean.square() - logvar.exp()).sum(1).mean()
        return reconstruction + kl, {'reconstruction_bce': reconstruction, 'kl': kl}
    reconstructed, latent = model(images)
    reconstruction = F.mse_loss(reconstructed, images)
    penalty = latent.abs().mean()
    loss = reconstruction + (CONFIG['sparsity_weight'] * penalty if name == 'SparseAE' else 0)
    return loss, {'reconstruction': reconstruction, 'latent_l1': penalty}

In [ ]:
# - Train Model
def train_model(name, data, classifier=None):
    constructors = {'AE': Autoencoder, 'SparseAE': Autoencoder, 'VAE': VAE, 'Classifier': Classifier, 'FrozenDecoder': decoder, 'DDPM': NoisePredictor}
    seed_everything()
    model = constructors[name]().to(DEVICE)
    diffusion = Diffusion(DEVICE, CONFIG['diffusion_steps'])
    checkpoint = OUTPUT / f'{name}.pt'
    if checkpoint.exists():
        if not (OUTPUT / f'{name}_selection.json').exists():
            raise RuntimeError(f'{name}: training sebelumnya belum lengkap. Arsipkan checkpoint parsial sebelum mengulang.')
        model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
        print(f'{name}: memakai checkpoint tersimpan', flush=True)
        return model.eval()
    learning_rate = 2e-4 if name == 'DDPM' else 1e-3
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history, best_loss, best_epoch = [], float('inf'), 0
    for epoch in range(1, CONFIG['epochs'][name] + 1):
        synchronize()
        started = time.perf_counter()
        model.train()
        sums = {}
        for images, labels in loader(data['train'], training=True):
            optimizer.zero_grad(set_to_none=True)
            loss, components = objective(name, model, images, labels, classifier, diffusion)
            if not torch.isfinite(loss):
                raise RuntimeError(f'Non-finite loss: {name}, epoch {epoch}')
            loss.backward()
            optimizer.step()
            for key, value in {'loss': loss, **components}.items():
                sums[key] = sums.get(key, 0) + value.detach() * len(images)
        synchronize()
        seconds = time.perf_counter() - started  # - Training saja, tanpa validation
        model.eval()
        validation = {}
        # - Validation memakai noise tetap dan tidak mengubah RNG training
        with torch.random.fork_rng(devices=[torch.cuda.current_device()] if DEVICE.type == 'cuda' else []):
            torch.manual_seed(31415)
            with torch.no_grad():
                for images, labels in loader(data['validation']):
                    loss, components = objective(name, model, images, labels, classifier, diffusion)
                    for key, value in {'loss': loss, **components}.items():
                        validation[key] = validation.get(key, 0) + value.detach() * len(images)
        record = {'epoch': epoch, 'seconds': seconds, **{key: float(value / len(data['train'])) for key, value in sums.items()}, **{'val_' + key: float(value / len(data['validation'])) for key, value in validation.items()}}
        history.append(record)
        if record['val_loss'] < best_loss:
            best_loss, best_epoch = record['val_loss'], epoch
            torch.save(model.state_dict(), checkpoint)
        pd.DataFrame(history).to_csv(OUTPUT / f'{name}_history.csv', index=False)
        print(f"{name} {epoch}/{CONFIG['epochs'][name]} loss={record['loss']:.5f} val={record['val_loss']:.5f} sec={seconds:.1f}", flush=True)
    (OUTPUT / f'{name}_selection.json').write_text(json.dumps({'rule': 'minimum validation objective', 'best_epoch': best_epoch, 'val_loss': best_loss}))
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    return model.eval()

In [ ]:
# - Train Gan
def train_gan(name, data):
    seed_everything()
    convolutional = name == 'DCGAN'
    generator = Generator(convolutional).to(DEVICE)
    discriminator = Discriminator(convolutional).to(DEVICE)
    checkpoint = OUTPUT / f'{name}.pt'
    if checkpoint.exists():
        if not (OUTPUT / f'{name}_selection.json').exists():
            raise RuntimeError(f'{name}: training sebelumnya belum lengkap. Arsipkan checkpoint parsial sebelum mengulang.')
        generator.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
        return generator.eval()
    if convolutional:
        for layer in list(generator.modules()) + list(discriminator.modules()):
            if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.normal_(layer.weight, 0, 0.02)
            elif isinstance(layer, nn.BatchNorm2d):
                nn.init.normal_(layer.weight, 1, 0.02)
                nn.init.zeros_(layer.bias)
    generator_optimizer = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    history = []
    for epoch in range(1, CONFIG['epochs'][name] + 1):
        synchronize()
        started = time.perf_counter()
        generator.train()
        discriminator.train()
        sums = torch.zeros(2, device=DEVICE)
        for images, _ in loader(data['train'], training=True):
            real = images * 2 - 1
            count = len(real)
            discriminator_optimizer.zero_grad(set_to_none=True)
            fake = generator(torch.randn(count, generator.latent_dim, device=DEVICE))
            real_logits, fake_logits = discriminator(real), discriminator(fake.detach())
            discriminator_loss = F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits)) + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits))
            discriminator_loss.backward()
            discriminator_optimizer.step()
            generator_optimizer.zero_grad(set_to_none=True)
            discriminator.requires_grad_(False)
            fake_logits = discriminator(fake)
            generator_loss = F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))
            if not torch.isfinite(generator_loss + discriminator_loss):
                raise RuntimeError(f'Non-finite GAN loss: {name}')
            generator_loss.backward()
            generator_optimizer.step()
            discriminator.requires_grad_(True)
            discriminator.train()
            sums += torch.stack([generator_loss.detach(), discriminator_loss.detach()]) * count
        synchronize()
        record = {'epoch': epoch, 'seconds': time.perf_counter() - started, 'generator_loss': float(sums[0] / len(data['train'])), 'discriminator_loss': float(sums[1] / len(data['train']))}
        history.append(record)
        pd.DataFrame(history).to_csv(OUTPUT / f'{name}_history.csv', index=False)
        print(f"{name} {epoch}/{CONFIG['epochs'][name]} G={record['generator_loss']:.4f} D={record['discriminator_loss']:.4f} sec={record['seconds']:.1f}", flush=True)
    torch.save(generator.state_dict(), checkpoint)
    torch.save(discriminator.state_dict(), OUTPUT / f'{name}_discriminator.pt')
    (OUTPUT / f'{name}_selection.json').write_text(json.dumps({'rule': 'fixed final epoch; no test-based selection', 'best_epoch': CONFIG['epochs'][name]}))
    return generator.eval()

In [ ]:
# - Train All
def train_all(data):
    models = {}
    for name in ['AE', 'SparseAE', 'VAE', 'Classifier']:
        models[name] = train_model(name, data)
    models['Classifier'].requires_grad_(False)
    models['FrozenDecoder'] = train_model('FrozenDecoder', data, models['Classifier'])
    for name in ['GAN', 'DCGAN']:
        models[name] = train_gan(name, data)
    models['DDPM'] = train_model('DDPM', data)
    return models

In [ ]:
# - Evaluation setup
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from torchmetrics.functional.image import structural_similarity_index_measure
from torchmetrics.image.fid import FrechetInceptionDistance

In [ ]:
# - Reconstruct
@torch.no_grad()
def reconstruct(name, model, images, classifier):
    if name in ['AE', 'SparseAE']:
        return model(images)[0]
    if name == 'VAE':
        return model.decoder(model.encode(images)[0])  # - Posterior mean untuk evaluation deterministik
    return model(classifier.encoder(images))

In [ ]:
# - Generate
@torch.no_grad()
def generate(name, model, count, diffusion):
    if name == 'DDPM':
        return diffusion.sample(model, count)
    if name == 'VAE':
        return model.decoder(torch.randn(count, 64, device=DEVICE))
    return (model(torch.randn(count, model.latent_dim, device=DEVICE)) + 1) / 2

In [ ]:
# - Reconstruction Metrics
@torch.no_grad()
def reconstruction_metrics(name, model, dataset, classifier):
    values = []
    for images, _ in loader(dataset):
        prediction = reconstruct(name, model, images, classifier)
        mse = (prediction - images).square().flatten(1).mean(1)
        psnr = -10 * torch.log10(mse.clamp_min(1e-12))
        ssim = structural_similarity_index_measure(prediction, images, data_range=1.0, reduction='none')
        values.append(torch.stack([mse, psnr, ssim], 1).cpu())
    values = torch.cat(values).numpy()
    pd.DataFrame(values, columns=['mse', 'psnr', 'ssim']).to_csv(OUTPUT / f'{name}_test_per_image.csv', index=False)
    return dict(zip(['test_mse', 'test_psnr', 'test_ssim'], values.mean(0).tolist()))

In [ ]:
# - Measure Latency
@torch.no_grad()
def measure_latency(function, repeats=10):
    for _ in range(3):
        function()
    times = []
    for _ in range(repeats):
        synchronize()
        started = time.perf_counter()
        function()
        synchronize()
        times.append((time.perf_counter() - started) * 1000)
    return {'latency_ms': float(np.median(times)), 'latency_p95_ms': float(np.percentile(times, 95)), 'latency_repeats': repeats}

In [ ]:
# - Evaluate
@torch.no_grad()
def evaluate(models, data):
    seed_everything(2026)
    diffusion = Diffusion(DEVICE, CONFIG['diffusion_steps'])
    classifier = models['Classifier']
    test_images, test_labels = data['test'].tensors
    accuracy = sum(int((classifier(images).argmax(1) == labels).sum()) for images, labels in loader(data['test'])) / len(test_images)
    summary = {'classifier_test_accuracy': accuracy, 'classifier_meets_88_percent': accuracy >= 0.88}
    rows = []
    for name in ['AE', 'SparseAE', 'VAE', 'GAN', 'DCGAN', 'DDPM', 'FrozenDecoder']:
        model = models[name].eval()
        history = pd.read_csv(OUTPUT / f'{name}_history.csv')
        row = {'model': name, 'latent_dim': 784 if name == 'DDPM' else (100 if name == 'DCGAN' else 64), 'parameters': sum(parameter.numel() for parameter in model.parameters()), 'training_seconds_per_epoch': history.seconds.mean(), 'epochs': len(history)}
        if name in ['AE', 'SparseAE', 'VAE', 'FrozenDecoder']:
            row.update(reconstruction_metrics(name, model, data['test'], classifier))
        if name in ['AE', 'SparseAE', 'FrozenDecoder']:
            row.update(measure_latency(lambda: reconstruct(name, model, test_images[:64], classifier)))
        else:
            row.update(measure_latency(lambda: generate(name, model, 64, diffusion), repeats=5 if name == 'DDPM' else 20))
        rows.append(row)
        print('Test metrics:', row, flush=True)
    # - FID memakai Inception-v3 2048-dim dari torch-fidelity
    torch.hub.set_dir(str(ROOT / 'data' / 'torch_cache'))
    # - Features FP32; akumulasi covariance TorchMetrics tetap FP64
    fid = FrechetInceptionDistance(feature=2048, normalize=False, reset_real_features=False, antialias=False).to(DEVICE)
    indices = np.random.default_rng(2026).choice(len(test_images), CONFIG['fid_samples'], replace=False)
    np.save(OUTPUT / 'fid_test_indices.npy', indices)
    def to_rgb(images):
        return (images.clamp(0, 1) * 255).round().to(torch.uint8).repeat(1, 3, 1, 1)
    for start in range(0, len(indices), 64):
        fid.update(to_rgb(test_images[indices[start:start + 64]]), real=True)
    print('Real FID features ready.', flush=True)
    for name in ['VAE', 'GAN', 'DCGAN', 'DDPM']:
        seed_everything(2026)
        fid.reset()
        generated = []
        counts = torch.zeros(10, device=DEVICE)
        confidences = []
        for start in range(0, CONFIG['fid_samples'], 64):
            images = generate(name, models[name], min(64, CONFIG['fid_samples'] - start), diffusion)
            generated.append(images.cpu())
            fid.update(to_rgb(images), real=False)
            probabilities = classifier(images).softmax(1)
            counts += torch.bincount(probabilities.argmax(1), minlength=10)
            confidences.append(probabilities.max(1).values)
        torch.save(torch.cat(generated), OUTPUT / f'{name}_samples.pt')
        score = float(fid.compute())
        distribution = counts / counts.sum()
        entropy = float(-(distribution * distribution.clamp_min(1e-12).log()).sum())
        next(row for row in rows if row['model'] == name).update({'fid': score, 'predicted_class_entropy': entropy, 'mean_classifier_confidence': float(torch.cat(confidences).mean())})
        summary[name + '_class_distribution'] = distribution.cpu().tolist()
        print(f'{name} FID ({CONFIG["fid_samples"]} samples) = {score:.4f}', flush=True)
        pd.DataFrame(rows).to_csv(OUTPUT / 'metrics.csv', index=False)
    # - Eksperimen proyeksi transpos dan struktur matriks
    first, second, head = classifier.encoder[1], classifier.encoder[3], classifier.head
    latent = classifier.encoder(test_images)
    transpose = torch.sigmoid(F.relu(latent @ second.weight) @ first.weight).reshape(-1, 1, 28, 28)
    summary['latent_transpose_test_mse'] = float((transpose - test_images).square().mean())
    summary['first_weight_rank'] = int(torch.linalg.matrix_rank(first.weight))
    summary['first_weight_nullity'] = 784 - summary['first_weight_rank']
    gram = first.weight.T @ first.weight
    summary['gram_identity_relative_error'] = float(torch.linalg.norm(gram - torch.eye(784, device=DEVICE)) / (784 ** 0.5))
    for name in ['AE', 'SparseAE']:
        hidden = models[name].encoder(test_images)
        summary[name + '_latent_mean_abs'] = float(hidden.abs().mean())
        summary[name + '_latent_zero_fraction'] = float((hidden < 1e-3).float().mean())
    summary['fid_protocol'] = {'feature': 2048, 'real_count': CONFIG['fid_samples'], 'fake_count': CONFIG['fid_samples'], 'real_source': 'fixed random subset of internal test', 'input': 'rounded uint8 RGB replication; torch-fidelity TensorFlow-compatible resize; antialias=False', 'feature_precision': 'FP32', 'covariance_precision': 'FP64', 'seed': 2026}
    (OUTPUT / 'summary.json').write_text(json.dumps(summary, indent=2))
    pd.DataFrame(rows).to_csv(OUTPUT / 'metrics.csv', index=False)
    return pd.DataFrame(rows), summary

In [ ]:
# - Plot Rows
def plot_rows(rows, labels, filename, column_titles=None):
    count = rows[0].shape[0]
    figure, axes = plt.subplots(len(rows), count, figsize=(count * 1.35 + 1, len(rows) * 1.45), squeeze=False)
    for row_index, (images, label) in enumerate(zip(rows, labels)):
        for column, image in enumerate(images):
            axes[row_index, column].imshow(image.squeeze().detach().cpu(), cmap='gray', vmin=0, vmax=1)
            axes[row_index, column].set_xticks([])
            axes[row_index, column].set_yticks([])
            if column == 0:
                axes[row_index, column].set_ylabel(label, fontsize=9)
            if row_index == 0 and column_titles:
                axes[row_index, column].set_title(column_titles[column], fontsize=9)
    figure.tight_layout()
    figure.savefig(OUTPUT / filename, dpi=220, bbox_inches='tight')
    plt.close(figure)

In [ ]:
# - Create Figures
@torch.no_grad()
def create_figures(models, data):
    seed_everything(777)
    images, labels = data['test'].tensors
    classifier = models['Classifier']
    selected = torch.stack([torch.where(labels == category)[0][0] for category in range(8)])
    originals = images[selected]
    plot_rows([originals] + [reconstruct(name, models[name], originals, classifier) for name in ['AE', 'SparseAE', 'VAE']], ['Original', 'Vanilla AE', 'Sparse AE', 'VAE'], 'figure1_reconstruction.png')
    figure, axes = plt.subplots(1, 4, figsize=(12, 3.3))
    for axis, name in zip(axes, ['VAE', 'GAN', 'DCGAN', 'DDPM']):
        samples = torch.load(OUTPUT / f'{name}_samples.pt', weights_only=True)
        axis.imshow(make_grid(samples[:16], nrow=4, padding=2, pad_value=1).permute(1, 2, 0))
        axis.set_title(name)
        axis.axis('off')
    figure.tight_layout()
    figure.savefig(OUTPUT / 'figure2_generation.png', dpi=220, bbox_inches='tight')
    plt.close(figure)
    first, second, head = classifier.encoder[1], classifier.encoder[3], classifier.head
    latent = classifier.encoder(originals)
    one_hot = F.one_hot(labels[selected], 10).float()
    class_projection = torch.sigmoid(F.relu(F.relu(one_hot @ head.weight) @ second.weight) @ first.weight).reshape(-1, 1, 28, 28)
    latent_projection = torch.sigmoid(F.relu(latent @ second.weight) @ first.weight).reshape(-1, 1, 28, 28)
    test_latent = classifier.encoder(images)
    train_latent = classifier.encoder(data['train'].tensors[0])
    noise = torch.randn(8, 64, device=DEVICE)
    diagnostic_samples = models['FrozenDecoder'](test_latent.mean(0) + noise * test_latent.std(0))
    train_samples = models['FrozenDecoder'](train_latent.mean(0) + noise * train_latent.std(0))
    plot_rows([originals, class_projection, latent_projection, models['FrozenDecoder'](latent), models['AE'](originals)[0], diagnostic_samples], ['Original', 'Class transpose', 'Latent transpose', 'Frozen decoder', 'Vanilla AE', 'MLP sampling*'], 'figure3_classifier.png')
    plot_rows([diagnostic_samples, train_samples], ['Test stats*', 'Train stats'], 'mlp_sampling_control.png')
    # - Endpoint AE/VAE berasal dari kelas 0 dan 9 pada validation set
    val_images, val_labels = data['validation'].tensors
    endpoint_images = torch.stack([val_images[torch.where(val_labels == category)[0][0]] for category in [0, 9]])
    alpha = torch.linspace(0, 1, 6, device=DEVICE)[:, None]
    rows, endpoint_info = [], {}
    for name in ['AE', 'SparseAE', 'VAE', 'GAN', 'DCGAN']:
        model = models[name]
        if name in ['AE', 'SparseAE']:
            endpoints = model.encoder(endpoint_images)
        elif name == 'VAE':
            endpoints = model.encode(endpoint_images)[0]
        else:
            # - GAN tanpa encoder: pilih prior endpoint dengan prediction kelas 0 dan 9
            candidates = torch.randn(4096, model.latent_dim, device=DEVICE)
            generated = torch.cat([(model(batch) + 1) / 2 for batch in candidates.split(256)])
            probabilities = classifier(generated).softmax(1)
            chosen = []
            for category in [0, 9]:
                matches = torch.where(probabilities.argmax(1) == category)[0]
                if len(matches) == 0:
                    raise RuntimeError(f'{name}: tidak ada endpoint predicted class {category}')
                chosen.append(matches[probabilities[matches, category].argmax()])
            endpoints = candidates[torch.stack(chosen)]
            endpoint_info[name] = {'predicted_classes': [0, 9], 'selection': 'highest classifier confidence among 4096 prior candidates; not ground truth', 'confidence': [float(probabilities[index, category]) for index, category in zip(chosen, [0, 9])]}
        interpolated = (1 - alpha) * endpoints[0] + alpha * endpoints[1]
        rows.append((model(interpolated) + 1) / 2 if name in ['GAN', 'DCGAN'] else model.decoder(interpolated))
    (OUTPUT / 'interpolation_protocol.json').write_text(json.dumps(endpoint_info, indent=2))
    plot_rows(rows, ['Vanilla AE', 'Sparse AE', 'VAE', 'Vanilla GAN', 'DCGAN'], 'figure4_interpolation.png', [f'alpha={value:.1f}' for value in np.linspace(0, 1, 6)])
    figure, axes = plt.subplots(2, 3, figsize=(12, 6))
    for axis, name in zip(axes.flatten(), ['AE', 'SparseAE', 'VAE', 'GAN', 'DCGAN', 'DDPM']):
        history = pd.read_csv(OUTPUT / f'{name}_history.csv')
        keys = ['generator_loss', 'discriminator_loss'] if name in ['GAN', 'DCGAN'] else ['loss', 'val_loss']
        for key in keys:
            axis.plot(history.epoch, history[key], label=key)
        axis.set(title=name, xlabel='Epoch', ylabel='Objective')
        axis.legend(fontsize=8)
        axis.grid(alpha=0.2)
    figure.tight_layout()
    figure.savefig(OUTPUT / 'figure5_loss.png', dpi=220, bbox_inches='tight')
    plt.close(figure)
    diffusion = Diffusion(DEVICE, CONFIG['diffusion_steps'])
    _, frames = diffusion.sample(models['DDPM'], 1, trajectory=True)
    plot_rows([torch.cat([(frame.clamp(-1, 1) + 1) / 2 for frame in frames])], ['DDPM'], 'figure5_trajectory.png', ['t=200', 't=159', 't=119', 't=79', 't=39', 't=0'])
    print('Five comparative figures saved.', flush=True)

In [ ]:
# - Dataset and training
data = prepare_data()
models = train_all(data)  # - Pakai checkpoint lengkap jika tersedia
print({split: len(dataset) for split, dataset in data.items()})

In [ ]:
# - Test evaluation
RUN_EVALUATION = False  # - Ubah True untuk menghitung ulang evaluation dari checkpoint
if RUN_EVALUATION or not (OUTPUT / 'metrics.csv').exists() or not (OUTPUT / 'summary.json').exists():
    metrics, summary = evaluate(models, data)
    create_figures(models, data)
else:
    metrics = pd.read_csv(OUTPUT / 'metrics.csv')
    summary = json.loads((OUTPUT / 'summary.json').read_text())
    print('Menampilkan evaluation tersimpan; training dan FID tidak dijalankan ulang.')
display(metrics.round(4))
print('Classifier test accuracy:', summary['classifier_test_accuracy'])

In [ ]:
# - Figure1 Reconstruction
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'figure1_reconstruction.png')))

In [ ]:
# - Figure2 Generation
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'figure2_generation.png')))

In [ ]:
# - Figure3 Classifier
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'figure3_classifier.png')))

In [ ]:
# - Figure4 Interpolation
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'figure4_interpolation.png')))

In [ ]:
# - Figure5 Loss
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'figure5_loss.png')))

In [ ]:
# - Figure5 Trajectory
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'figure5_trajectory.png')))

In [ ]:
# - Mlp Sampling Control
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'mlp_sampling_control.png')))

## Interpretation notes
Metrik dan figur berasal dari eksperimen local. Sampling MLP dengan statistik test bersifat diagnostik. FID memakai 3.000 sampel per distribusi; latency GPU desktop bukan latency mobile. Analisis lengkap dan batasan ada pada laporan.